In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch

from transformers import (AutoTokenizer,AutoModelForCausalLM)

In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
question = "What is the capital of Canada?"
messages = [
    {
        "role": "user",
        "content": question
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

In [ ]:
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print(answer)

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm

# 路径
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "Data" / "TruthfulQA_preset.csv"
OUTPUT_PATH = PROJECT_ROOT / "Results" / "Phi3_Responses.csv"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# 读取数据
df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
def generate_phi3_answer(question, max_new_tokens=150):
    messages = [
        {
            "role": "user",
            "content": str(question)
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

In [ ]:
question = df.loc[0, "Question"]

answer = generate_phi3_answer(question)

print("Question:")
print(question)

print("\nPhi-3 Answer:")
print(answer)

In [ ]:
phi3_answers = []

for i, question in enumerate(tqdm(df["Question"], desc="Generating Phi-3 answers")):
    try:
        answer = generate_phi3_answer(question)
    except Exception as e:
        print(f"Error at row {i}: {e}")
        answer = ""

    phi3_answers.append(answer)

    if (i + 1) % 10 == 0:
        temp_df = df.iloc[:i + 1].copy()
        temp_df["Phi3_Answer"] = phi3_answers
        temp_df.to_csv(OUTPUT_PATH, index=False)

        print(f"Saved {i + 1}/{len(df)} answers")

In [ ]:
df["Phi3_Answer"] = phi3_answers

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Finished!")
print("Saved to:", OUTPUT_PATH)

In [ ]:
df.head()